In [1]:
# Run if needed
#pip install -q kagglehub[pandas-datasets]
#%pip install pyspark
#%pip install matplotlib
#%pip install seaborn scipy

In [2]:
# Import necessary libraries for data analysis and visualization
import pandas as pd   # For data manipulation and analysis
import numpy as np    # For numerical operations
import matplotlib.pyplot as plt  # For creating visualizations
import seaborn as sns     # For statistical data visualization
from scipy import stats   # For statistical functions and tests
from pyspark.sql import SparkSession
import tempfile
import urllib.request
from functools import reduce
from operator import add
from pyspark.sql import functions as F

# 1. Initialize SparkSession
spark = SparkSession.builder \
    .appName("GenderPayGapEurope") \
    .getOrCreate()

Bagaimana perkembangan gender pay gap di eropa 

1. Bagaimana trend pay gap di Eropa seiring waktu? 
2. Berdasarkan Industri, bagaimana trend pay gap?
3. Apakah tingkat edukasi wanita berpengaruh dengan pay gap? (harus join, terus pake statistik)
4. Seberapa besar persentase gap employment antara gender?
5. Seberapa besar persentase wanita masuk menjabat dalam manajemen suatu perusahaan?

Kesimpulan: berasarkan tahun, pay gap semakin menurun. Melihat industri private dan publik, pay gap naik di private. Negara dengan GDP tinggi, pay gap rendah

Semua dataset berdasarkan tahun 2015 -2024

Data set : Gender Pay Gap - Europe 
Bagaimana trend seiring waktu tentang pay gap
Bagaimana trend berdasarkan industri tentang pay gap
Apakah tingkat GDP mempengaruhi pay gap, (Harus ada standar klasifikasi)
Bisa dibuat analisa statistik dan graphs
Link : (https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/earn_gr_gpgr2/1.0/*.*.*.*?c[freq]=A&c[unit]=PC&c[nace_r2]=B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,UK,ME,MK,AL,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name)

Data set : Employment gap
Menjelaskan gap employment rate antara pria dan wanita
Seberapa besar persentase gap employment antara gender?
Nilai positif berarti jumlah pria lebih bayak, nilai negatif berarti wanita lebih banyak 
Link (https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_05_30/1.0/*.*.*.*.*?c[freq]=A&c[wstatus]=EMP,EMP_PT,EMP_TEMP,UEMP_PT&c[age]=Y20-64&c[unit]=PC_PNT&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,BA,ME,MK,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name)

Data set : Women in senior posistion management
Menjelaskan berapa banyak persentase wanita menjabat di bagian management suatu perusahaan
Seberapa besar persentase wanita masuk menjabat dalam manajemen suatu perusahaan?
Semakin tinggi persentase, semakin tinggi jumlah wanita yang menjabat sebagai manajemen di suatu persuahaan 
Link (https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_05_61/1.0/*.*.*.*.*?c[freq]=A&c[prof_pos]=NEXEC,NEXEC_EXEC&c[sex]=F&c[unit]=PC&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,BA,ME,MK,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name)

Data set : Female aged 25-34 with tertiary educational (kuliah) attainment level
Berapa besar presentase wanita umur 25 - 34 mengenyam pendidikan kampus (kuliah)
Apakah tingkat edukasi wanita berpengaruh dengan pay gap? (harus join, terus pake statistik)
Persentasenya dibandingkan dengan keseluruhan populasi pada beacket umur 25 - 34
Link (https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_04_20/1.0/*.*.*.*.*.*?c[freq]=A&c[sex]=F&c[age]=Y25-34&c[unit]=PC&c[isced11]=ED5-8&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,BA,ME,MK,RS,TR,UK&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name)


LIMITASI DATA

# 1. Extract & Load Data to Pyspark

In [3]:
def read_csv(file_name):
    # If input is an HTTP/HTTPS URL, download to a local temp .csv.gz file first
    if file_name.startswith(("http://", "https://")):
        suffix = ".csv.gz" if "compress=true" in file_name else ".csv"
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.close()
        urllib.request.urlretrieve(file_name, tmp.name)
        local_path = tmp.name
    else:
        local_path = file_name

    return spark.read.csv(local_path, header=True, inferSchema=True, escape='"')

def to_snake_case(df):
    new_names = [
        c.strip().lower()
         .replace(' ', '_')
         .replace('-', '_')
         .replace('/', '_')
         .replace('.', '')
         .replace('(', '')
         .replace(')', '')
        for c in df.columns
    ]
    return df.toDF(*new_names)

wage_gap = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/earn_gr_gpgr2/1.0/*.*.*.*?c[freq]=A&c[unit]=PC&c[nace_r2]=B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,UK,ME,MK,AL,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))
emp_gap = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_05_30/1.0/*.*.*.*.*?c[freq]=A&c[wstatus]=EMP,EMP_PT,EMP_TEMP,UEMP_PT&c[age]=Y20-64&c[unit]=PC_PNT&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,BA,ME,MK,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))
women_mang = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_05_61/1.0/*.*.*.*.*?c[freq]=A&c[prof_pos]=NEXEC,NEXEC_EXEC&c[sex]=F&c[unit]=PC&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,BA,ME,MK,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))
women_college = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_04_20/1.0/*.*.*.*.*.*?c[freq]=A&c[sex]=F&c[age]=Y25-34&c[unit]=PC&c[isced11]=ED5-8&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,BA,ME,MK,RS,TR,UK&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))


# 2. Data Exploration

In [4]:
wage_gap.show()

+---------+--------------------+--------------------+----+--------------+----+---------------+-------+--------------------------------------------------------------------------------------+---+-----------------------------+-----------+----+---------+-----------------+--------+------------------------------------+-----------+---------------------------+
|structure|        structure_id|      structure_name|freq|time_frequency|unit|unit_of_measure|nace_r2|statistical_classification_of_economic_activities_in_the_european_community_nace_rev_2|geo|geopolitical_entity_reporting|time_period|time|obs_value|observation_value|obs_flag|observation_status_flag_v2_structure|conf_status|confidentiality_status_flag|
+---------+--------------------+--------------------+----+--------------+----+---------------+-------+--------------------------------------------------------------------------------------+---+-----------------------------+-----------+----+---------+-----------------+--------+-------------

In [5]:
emp_gap.show()

+---------+--------------------+--------------------+----+--------------+-------+----------------------------------+------+-------------------+------+----------------+---+-----------------------------+-----------+----+---------+-----------------+--------+------------------------------------+-----------+---------------------------+
|structure|        structure_id|      structure_name|freq|time_frequency|wstatus|labour_force_and_employment_status|   age|          age_class|  unit| unit_of_measure|geo|geopolitical_entity_reporting|time_period|time|obs_value|observation_value|obs_flag|observation_status_flag_v2_structure|conf_status|confidentiality_status_flag|
+---------+--------------------+--------------------+----+--------------+-------+----------------------------------+------+-------------------+------+----------------+---+-----------------------------+-----------+----+---------+-----------------+--------+------------------------------------+-----------+---------------------------+
|

In [6]:
women_mang.show()

+---------+--------------------+--------------------+----+--------------+--------+---------------------+----+-------+----+---------------+---+-----------------------------+-----------+----+---------+-----------------+--------+------------------------------------+-----------+---------------------------+
|structure|        structure_id|      structure_name|freq|time_frequency|prof_pos|professional_position|sex7|   sex8|unit|unit_of_measure|geo|geopolitical_entity_reporting|time_period|time|obs_value|observation_value|obs_flag|observation_status_flag_v2_structure|conf_status|confidentiality_status_flag|
+---------+--------------------+--------------------+----+--------------+--------+---------------------+----+-------+----+---------------+---+-----------------------------+-----------+----+---------+-----------------+--------+------------------------------------+-----------+---------------------------+
| dataflow|ESTAT:SDG_05_61(1.0)|Positions held by...|   A|        Annual|   NEXEC| Non-e

In [7]:
women_college.show()

+---------+--------------------+--------------------+----+--------------+----+-------+------+-------------------+----+---------------+-------+-------------------------------------------------------------+---+-----------------------------+-----------+----+---------+-----------------+--------+------------------------------------+-----------+---------------------------+
|structure|        structure_id|      structure_name|freq|time_frequency|sex5|   sex6|   age|          age_class|unit|unit_of_measure|isced11|international_standard_classification_of_education_isced_2011|geo|geopolitical_entity_reporting|time_period|time|obs_value|observation_value|obs_flag|observation_status_flag_v2_structure|conf_status|confidentiality_status_flag|
+---------+--------------------+--------------------+----+--------------+----+-------+------+-------------------+----+---------------+-------+-------------------------------------------------------------+---+-----------------------------+-----------+----+-----

## Rows & Column

In [8]:
print(f'Table wage_gap contains {wage_gap.count()} rows and {len(wage_gap.columns)} columns')
print(f'Table emp_gap contains {emp_gap.count()} rows and {len(emp_gap.columns)} columns')
print(f'Table Order women_mang contains {women_mang.count()} rows and {len(women_mang.columns)} columns')
print(f'Table Order women_college contains {women_college.count()} rows and {len(women_college.columns)} columns')

Table wage_gap contains 4829 rows and 19 columns
Table emp_gap contains 1360 rows and 21 columns
Table Order women_mang contains 664 rows and 21 columns
Table Order women_college contains 345 rows and 23 columns


## Data Type

In [9]:
wage_gap.printSchema(),
emp_gap.printSchema(),
women_mang.printSchema(),
women_college.printSchema()


root
 |-- structure: string (nullable = true)
 |-- structure_id: string (nullable = true)
 |-- structure_name: string (nullable = true)
 |-- freq: string (nullable = true)
 |-- time_frequency: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- unit_of_measure: string (nullable = true)
 |-- nace_r2: string (nullable = true)
 |-- statistical_classification_of_economic_activities_in_the_european_community_nace_rev_2: string (nullable = true)
 |-- geo: string (nullable = true)
 |-- geopolitical_entity_reporting: string (nullable = true)
 |-- time_period: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- obs_value: double (nullable = true)
 |-- observation_value: string (nullable = true)
 |-- obs_flag: string (nullable = true)
 |-- observation_status_flag_v2_structure: string (nullable = true)
 |-- conf_status: string (nullable = true)
 |-- confidentiality_status_flag: string (nullable = true)

root
 |-- structure: string (nullable = true)
 |-- structure_id: 

## Statistical Summary

In [10]:
wage_gap.describe().show(),
emp_gap.describe().show(),
women_mang.describe().show(),
women_college.describe().show()

+-------+---------+--------------------+--------------------+----+--------------+----+---------------+-------+--------------------------------------------------------------------------------------+----+-----------------------------+-----------------+----+------------------+-----------------+--------+------------------------------------+-----------+---------------------------+
|summary|structure|        structure_id|      structure_name|freq|time_frequency|unit|unit_of_measure|nace_r2|statistical_classification_of_economic_activities_in_the_european_community_nace_rev_2| geo|geopolitical_entity_reporting|      time_period|time|         obs_value|observation_value|obs_flag|observation_status_flag_v2_structure|conf_status|confidentiality_status_flag|
+-------+---------+--------------------+--------------------+----+--------------+----+---------------+-------+--------------------------------------------------------------------------------------+----+-----------------------------+----------

## Duplicate & Missing Value

In [11]:
total_rows_wage_gap = wage_gap.count()
distinct_rows_wage_gap = wage_gap.drop_duplicates().count()
duplicate_rows_wage_gap = total_rows_wage_gap - distinct_rows_wage_gap

print(f"Total Rows: {total_rows_wage_gap}")
print(f"Distinct Rows: {distinct_rows_wage_gap}")
print(f"Duplicate Rows: {duplicate_rows_wage_gap}")

Total Rows: 4829
Distinct Rows: 4829
Duplicate Rows: 0


In [12]:
print('Missing Value for table wage_gap:')

for c in wage_gap.columns:
    print(c, wage_gap[wage_gap[c].isNull()].count())

Missing Value for table wage_gap:
structure 0
structure_id 0
structure_name 0
freq 0
time_frequency 0
unit 0
unit_of_measure 0
nace_r2 0
statistical_classification_of_economic_activities_in_the_european_community_nace_rev_2 0
geo 0
geopolitical_entity_reporting 0
time_period 0
time 4829
obs_value 33
observation_value 4829
obs_flag 3493
observation_status_flag_v2_structure 3493
conf_status 4796
confidentiality_status_flag 4796


Tidak ada missing value 


In [13]:
total_rows_emp_gap = emp_gap.count()
distinct_rows_emp_gap = emp_gap.drop_duplicates().count()
duplicate_rows_emp_gap = total_rows_emp_gap - distinct_rows_emp_gap

print(f"Total Rows: {total_rows_emp_gap}")
print(f"Distinct Rows: {distinct_rows_emp_gap}")
print(f"Duplicate Rows: {duplicate_rows_emp_gap}")

Total Rows: 1360
Distinct Rows: 1360
Duplicate Rows: 0


In [44]:
print('Missing Value for table emp_gap:')

for c in emp_gap.columns:
    print(c, emp_gap[emp_gap[c].isNull()].count())

Missing Value for table emp_gap:
structure 0
structure_id 0
structure_name 0
freq 0
time_frequency 0
wstatus 0
labour_force_and_employment_status 0
age 0
age_class 0
unit 0
unit_of_measure 0
geo 0
geopolitical_entity_reporting 0
time_period 0
time 1360
obs_value 7
observation_value 1360
obs_flag 1237
observation_status_flag_v2_structure 1237
conf_status 1360
confidentiality_status_flag 1360


Columns that will be used:\
geopolitical_entity_(reporting) 0\
time_period 0\
obs_value 5


In [15]:
total_rows_women_mang = women_mang.count()
distinct_rows_women_mang = women_mang.drop_duplicates().count()
duplicate_rows_women_mang = total_rows_women_mang - distinct_rows_women_mang

print(f"Total Rows: {total_rows_women_mang}")
print(f"Distinct Rows: {distinct_rows_women_mang}")
print(f"Duplicate Rows: {duplicate_rows_women_mang}")

Total Rows: 664
Distinct Rows: 664
Duplicate Rows: 0


In [16]:
print('Missing Value for table women_mang:')

for c in women_mang.columns:
    print(c, women_mang[women_mang[c].isNull()].count())

Missing Value for table women_mang:
structure 0
structure_id 0
structure_name 0
freq 0
time_frequency 0
prof_pos 0
professional_position 0
sex7 0
sex8 0
unit 0
unit_of_measure 0
geo 0
geopolitical_entity_reporting 0
time_period 0
time 664
obs_value 0
observation_value 664
obs_flag 664
observation_status_flag_v2_structure 664
conf_status 664
confidentiality_status_flag 664


Columns that will be used:\
geopolitical_entity_(reporting) 0\
time_period 0\
obs_value 0

In [17]:
total_rows_women_college = women_college.count()
distinct_rows_women_college = women_college.drop_duplicates().count()
duplicate_rows_women_college = total_rows_women_college - distinct_rows_women_college

print(f"Total Rows: {total_rows_women_college}")
print(f"Distinct Rows: {distinct_rows_women_college}")
print(f"Duplicate Rows: {duplicate_rows_women_college}")

Total Rows: 345
Distinct Rows: 345
Duplicate Rows: 0


In [18]:
print('Missing Value for table women_college:')

for c in women_college.columns:
    print(c, women_college[women_college[c].isNull()].count())

Missing Value for table women_college:
structure 0
structure_id 0
structure_name 0
freq 0
time_frequency 0
sex5 0
sex6 0
age 0
age_class 0
unit 0
unit_of_measure 0
isced11 0
international_standard_classification_of_education_isced_2011 0
geo 0
geopolitical_entity_reporting 0
time_period 0
time 345
obs_value 0
observation_value 345
obs_flag 293
observation_status_flag_v2_structure 293
conf_status 345
confidentiality_status_flag 345


Columns that will be used:\
geopolitical_entity_(reporting) 0\
time_period 0\
obs_value 0

# 3. Transform

## Batch Cleaning

Batch cleaning will clean the data per table, usually cleaning will be done by:
1. Drop duplicate 
2. Capitalize and trim white space for string column
3. For column with number value, i will round it into 2 and cast as float
4. Handling Missing value: if string, filled with 'Unknown'. If number, filled with '0'

In [19]:
wage_gap_clean = (
    wage_gap.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.col("statistical_classification_of_economic_activities_in_the_european_community_nace_rev_2").alias("industry"),
        F.round(F.col("obs_value"),2).alias("wage_gap_rate(%)")
    )
)

wage_gap_clean.show(truncate=False)

+-----------+----+--------------------------------------------------------------------+----------------+
|country    |year|industry                                                            |wage_gap_rate(%)|
+-----------+----+--------------------------------------------------------------------+----------------+
|Romania    |2022|Manufacturing                                                       |21.0            |
|Slovenia   |2021|Manufacturing                                                       |11.5            |
|Italy      |2018|Water supply; sewerage, waste management and remediation activities |4.9             |
|Cyprus     |2022|Wholesale and retail trade; repair of motor vehicles and motorcycles|23.6            |
|Czechia    |2016|Wholesale and retail trade; repair of motor vehicles and motorcycles|25.4            |
|Iceland    |2021|Transportation and storage                                          |13.8            |
|Latvia     |2023|Transportation and storage           

In [36]:
emp_gap_clean = (
    emp_gap.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.round(F.col("obs_value"),2).alias("employment_gap_rate(%)")
    ).fillna(0)
)

emp_gap_clean.show()

+----------+----+----------------------+
|   country|year|employment_gap_rate(%)|
+----------+----+----------------------+
|  Bulgaria|2020|                   8.3|
|     Spain|2021|                  10.8|
|Montenegro|2015|                  10.3|
|    Cyprus|2017|                  -6.6|
|   Estonia|2018|                  -7.6|
|    Poland|2018|                  -5.9|
|   Belgium|2022|                  -2.2|
|   Germany|2019|                  -0.2|
|    Sweden|2021|                  -3.9|
|    Greece|2015|                  -3.5|
|   Iceland|2023|                  -1.0|
|   Germany|2015|                 -37.7|
|     Spain|2024|                 -14.8|
|     Italy|2019|                  -2.3|
|    Poland|2021|                  -2.2|
|   Finland|2024|                  -1.8|
|  Slovenia|2018|                   7.1|
|    Greece|2023|                  -7.8|
|    Norway|2016|                 -22.0|
|   Hungary|2023|                  -0.3|
+----------+----+----------------------+
only showing top

In [21]:
women_mang_clean = (
    women_mang.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.round(F.col("obs_value"),2).alias("women_management_rate(%)")
    )
)

women_mang_clean.show()

+----------+----+------------------------+
|   country|year|women_management_rate(%)|
+----------+----+------------------------+
|    Cyprus|2019|                     6.9|
|   Belgium|2019|                    38.6|
|   Croatia|2021|                    20.8|
|Montenegro|2017|                    25.0|
|   Romania|2015|                    11.1|
|  Slovakia|2016|                    12.5|
|   Türkiye|2019|                    20.3|
|    Latvia|2024|                    27.8|
|Montenegro|2016|                    23.3|
|   Ireland|2023|                    43.9|
|Montenegro|2021|                    20.3|
|     Malta|2018|                    10.6|
|   Hungary|2023|                    11.8|
|     Italy|2018|                    42.9|
|Luxembourg|2022|                    28.6|
|Luxembourg|2023|                    26.8|
|    Poland|2020|                    22.8|
|   Türkiye|2015|                    10.2|
|   Austria|2022|                    27.0|
|    France|2022|                    44.8|
+----------

In [22]:
women_college_clean = (
    women_college.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.round(F.col("obs_value"),2).alias("women_college_rate(%)")
    )
)

women_college_clean.show()

+--------------------+----+---------------------+
|             country|year|women_college_rate(%)|
+--------------------+----+---------------------+
|             Belgium|2015|                 49.2|
|             Türkiye|2024|                 46.5|
|             Belgium|2023|                 57.3|
|              Latvia|2021|                 55.4|
|          Montenegro|2020|                 45.0|
|              Sweden|2024|                 63.3|
|            Slovenia|2017|                 56.3|
|            Slovenia|2021|                 60.8|
|             Türkiye|2016|                 28.7|
|Bosnia And Herzeg...|2021|                 32.9|
|           Lithuania|2019|                 66.1|
|              Norway|2021|                 64.5|
|             Belgium|2024|                 56.9|
|          Luxembourg|2016|                 54.4|
|            Bulgaria|2024|                 49.2|
|              France|2019|                 52.3|
|             Romania|2024|                 26.3|


## Cleaning Code Compilation

def read_csv(file_name):
    # If input is an HTTP/HTTPS URL, download to a local temp .csv.gz file first
    if file_name.startswith(("http://", "https://")):
        suffix = ".csv.gz" if "compress=true" in file_name else ".csv"
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.close()
        urllib.request.urlretrieve(file_name, tmp.name)
        local_path = tmp.name
    else:
        local_path = file_name

    return spark.read.csv(local_path, header=True, inferSchema=True, escape='"')

def to_snake_case(df):
    new_names = [
        c.strip().lower()
         .replace(' ', '_')
         .replace('-', '_')
         .replace('/', '_')
         .replace('.', '')
         .replace('(', '')
         .replace(')', '')
        for c in df.columns
    ]
    return df.toDF(*new_names)

wage_gap = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/earn_gr_gpgr2/1.0/*.*.*.*?c[freq]=A&c[unit]=PC&c[nace_r2]=B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,UK,ME,MK,AL,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))
emp_gap = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_05_30/1.0/*.*.*.*.*?c[freq]=A&c[wstatus]=EMP,EMP_PT,EMP_TEMP,UEMP_PT&c[age]=Y20-64&c[unit]=PC_PNT&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,BA,ME,MK,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))
women_mang = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_05_61/1.0/*.*.*.*.*?c[freq]=A&c[prof_pos]=NEXEC,NEXEC_EXEC&c[sex]=F&c[unit]=PC&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,BA,ME,MK,RS,TR&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))
women_college = to_snake_case(read_csv('https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/sdg_04_20/1.0/*.*.*.*.*.*?c[freq]=A&c[sex]=F&c[age]=Y25-34&c[unit]=PC&c[isced11]=ED5-8&c[geo]=BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE,IS,NO,CH,BA,ME,MK,RS,TR,UK&c[TIME_PERIOD]=2024,2023,2022,2021,2020,2019,2018,2017,2016,2015&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name'))

wage_gap_clean = (
    wage_gap.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.col("statistical_classification_of_economic_activities_in_the_european_community_nace_rev_2").alias("industry"),
        F.round(F.col("obs_value"),2).alias("wage_gap_rate(%)")
    )
)

emp_gap_clean = (
    emp_gap.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.round(F.col("obs_value"),2).alias("employment_gap_rate(%)")
    )fillna(0)
)


women_mang_clean = (
    women_mang.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.round(F.col("obs_value"),2).alias("women_management_rate(%)")
    )
)

women_college_clean = (
    women_college.dropDuplicates().select(
        F.initcap(F.trim(F.col("geopolitical_entity_reporting"))).alias("country"),
        F.col("time_period").alias("year"),
        F.round(F.col("obs_value"),2).alias("women_college_rate(%)")
    )
)


spark.stop()

# Data Analysis

## 1. Bagaimana trend pay gap di Eropa seiring waktu? 

In [23]:
yearly_trend = (wage_gap_clean.groupBy("year")
    .agg(F.round(F.mean("wage_gap_rate(%)"), 2).alias("wage_gap_rate(%)"))
    .orderBy("year")
)

yearly_trend.show()

+----+----------------+
|year|wage_gap_rate(%)|
+----+----------------+
|2015|           13.51|
|2016|           13.13|
|2017|           13.05|
|2018|           12.17|
|2019|           12.25|
|2020|           11.76|
|2021|           11.41|
|2022|            10.9|
|2023|           10.83|
|2024|            10.6|
+----+----------------+



Berdasarkan data trend, pay gap terdapat fluktuasi pada besaran persentase pay gap di Eropa dalam kurun waktu 2010-2021.\
Fluktuasi menujukkan trend penurunan setiap tahun dengan kenaikan kecil pada 2017 sebesar 0.11%.\
Tahun 2015-2016 menunjukkan penurunan pay gap yang paling besar yaitu dari 13.96% menjadi 13.3% atau menurun 0.66%\
Overall, semenjak tahun 2010 hingga 2021 terdapat penurun gap pada pay gender perempuan sebesar 3.39%

Conclusion\
Pay Gap di Eropa semakin menurun setiap tahunnya sejak tahun 2010 sebesar 15.57% menjadi 12.18% pada tahun 2021

## 2. Berdasarkan Industri, bagaimana trend pay gap?

In [24]:
indust_trend = (
    wage_gap_clean.groupBy("industry").agg(
        F.round(F.mean("wage_gap_rate(%)"), 2).alias("wage_gap_rate(%)"),
    )
    .orderBy("wage_gap_rate(%)", ascending=False)
)

indust_trend.show(truncate=False)

+--------------------------------------------------------------------+----------------+
|industry                                                            |wage_gap_rate(%)|
+--------------------------------------------------------------------+----------------+
|Financial and insurance activities                                  |26.25           |
|Wholesale and retail trade; repair of motor vehicles and motorcycles|19.08           |
|Information and communication                                       |18.83           |
|Professional, scientific and technical activities                   |17.81           |
|Manufacturing                                                       |17.3            |
|Human health and social work activities                             |17.12           |
|Arts, entertainment and recreation                                  |16.51           |
|Real estate activities                                              |12.2            |
|Education                      

## 3. Apakah tingkat edukasi wanita berpengaruh dengan pay gap?

In [25]:
women_college_trend = (women_college_clean.groupBy("year")
    .agg(F.round(F.mean("women_college_rate(%)"), 2).alias("women_college_rate(%)"))
    .orderBy("year")
)

women_college_trend.show()

+----+---------------------+
|year|women_college_rate(%)|
+----+---------------------+
|2015|                45.51|
|2016|                46.43|
|2017|                47.27|
|2018|                48.12|
|2019|                49.25|
|2020|                50.02|
|2021|                50.71|
|2022|                50.96|
|2023|                51.38|
|2024|                52.64|
+----+---------------------+



In [26]:
edu_pay_gap = women_college_trend.join(yearly_trend, on="year", how="inner")
edu_pay_gap.show()

+----+---------------------+----------------+
|year|women_college_rate(%)|wage_gap_rate(%)|
+----+---------------------+----------------+
|2018|                48.12|           12.17|
|2015|                45.51|           13.51|
|2023|                51.38|           10.83|
|2022|                50.96|            10.9|
|2019|                49.25|           12.25|
|2020|                50.02|           11.76|
|2016|                46.43|           13.13|
|2024|                52.64|            10.6|
|2017|                47.27|           13.05|
|2021|                50.71|           11.41|
+----+---------------------+----------------+



## 4. Seberapa besar persentase gap employment antara gender?

In [38]:
emp_gap_trend = (
    emp_gap_clean.filter(F.col("employment_gap_rate(%)") != 0).groupBy("year").agg(
        F.round(F.mean("employment_gap_rate(%)"), 2).alias("employment_gap_rate(%)"),
    ).orderBy("year", ascending=False)
)

emp_gap_trend.show(truncate=False)

+----+----------------------+
|year|employment_gap_rate(%)|
+----+----------------------+
|2024|-1.65                 |
|2023|-1.67                 |
|2022|-1.67                 |
|2021|-1.69                 |
|2020|-1.75                 |
|2019|-1.8                  |
|2018|-1.77                 |
|2017|-1.85                 |
|2016|-1.91                 |
|2015|-1.97                 |
+----+----------------------+



## 5. Seberapa besar persentase wanita masuk menjabat dalam manajemen suatu perusahaan?

In [41]:
women_mang_trend = (
    women_mang_clean.groupBy("year").agg(
        F.round(F.mean("women_management_rate(%)"), 2).alias("women_management_rate(%)"),
    )
    .orderBy("women_management_rate(%)", ascending=False)
)

women_mang_trend.show(truncate=False)

+----+------------------------+
|year|women_management_rate(%)|
+----+------------------------+
|2024|31.26                   |
|2023|29.1                    |
|2022|28.05                   |
|2021|27.01                   |
|2020|25.63                   |
|2019|24.92                   |
|2018|23.28                   |
|2017|22.74                   |
|2016|21.54                   |
|2015|20.25                   |
+----+------------------------+

